# 01 — Does sleep amount / consistency / recent history predict HRV and resting heart rate?

**Question:** ignoring exercise entirely for now, does last night's sleep duration,
this stretch's sleep consistency, and the trailing 7 days of sleep history predict
that morning's HRV (`hrv_rmssd_milli`) and resting heart rate (`resting_heart_rate`)?

Reminder on the pairing (see `src/clean.py`): each row's `resting_heart_rate` and
`hrv_rmssd_milli` come from the sleep at the *end* of that row's cycle — i.e. the
night following that row's `date`, measured on waking. So "last night's sleep"
for a given row's HRV/RHR **is that same row's own sleep columns** — no shifting
needed for the same-night relationship. The 7-day rolling average, however,
deliberately **excludes** the current night, since it's meant to capture
cumulative sleep debt built up *before* that night, not the night itself.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

df = pd.read_csv("../data/daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
print(f"{len(df)} days, {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## Build the sleep variables

- `total_sleep_hours` — actual time asleep (light + deep/slow-wave + REM), **not**
  `hours_in_bed` (which also counts time spent awake in bed).
- `sleep_consistency_pct` — already computed by WHOOP, used as-is.
- `prior_5d_avg_sleep_hours` — rolling mean of `total_sleep_hours` over the
  trailing 5 nights, **shifted so it excludes the current night** (it's "how
  much sleep debt had built up walking into tonight," not tonight itself).
  Strict window: a night only gets a value once a full 5 real prior nights
  exist (`min_periods=5`) - no partial-window averages.

In [ ]:
df["total_sleep_hours"] = df["hours_light_sleep"] + df["hours_deep_sleep"] + df["hours_rem_sleep"]

# shift(1) first so the rolling window is the 5 nights BEFORE tonight, not including it.
# min_periods=5 (== window) makes this strict: no value until a full 5 real prior nights exist.
df["prior_5d_avg_sleep_hours"] = (
    df["total_sleep_hours"].shift(1).rolling(window=5, min_periods=5).mean()
)

sleep_vars = ["total_sleep_hours", "sleep_consistency_pct", "sleep_efficiency_pct", "prior_5d_avg_sleep_hours"]
outcome_vars = ["hrv_rmssd_milli", "resting_heart_rate"]

df[sleep_vars + outcome_vars].describe()

## Correlation matrix

Note: this drops any row missing *any* sleep variable (so n=266, matching the smallest-coverage variable, prior_5d_avg_sleep_hours), while the per-pair table below drops NaNs independently per pair (so some rows show n=270). That is why the r values differ slightly between the two — both are correct, they just use slightly different row sets.

Note: this drops any row missing *any* sleep variable (so n=266, matching the smallest-coverage variable, prior_5d_avg_sleep_hours), while the per-pair table below drops NaNs independently per pair (so some rows show n=270). That is why the r values differ slightly between the two - both are correct, they just use slightly different row sets.

In [ ]:
corr_df = df[sleep_vars + outcome_vars].dropna()
print(f"n = {len(corr_df)} days with complete data")
corr_df.corr(numeric_only=True).loc[sleep_vars, outcome_vars]

## Correlation + significance for each sleep variable vs each outcome

Pearson r alone doesn't tell you whether a relationship is likely real vs. noise —
the p-value does that (rule of thumb: p < 0.05 means "unlikely to be random chance
given this sample size"). Both matter; report both, not just r.

In [ ]:
results = []
for sv in sleep_vars:
    for ov in outcome_vars:
        pair = df[[sv, ov]].dropna()
        r, p = pearsonr(pair[sv], pair[ov])
        results.append({"sleep_var": sv, "outcome": ov, "n": len(pair), "r": round(r, 3), "p_value": round(p, 4)})

results_df = pd.DataFrame(results)
results_df

## Scatter plots: each sleep variable vs. each outcome

In [ ]:
fig, axes = plt.subplots(len(sleep_vars), len(outcome_vars), figsize=(5, 14))

for i, sv in enumerate(sleep_vars):
    for j, ov in enumerate(outcome_vars):
        ax = axes[i, j]
        pair = df[[sv, ov]].dropna()
        ax.scatter(pair[sv], pair[ov], alpha=0.4, s=15)
        if len(pair) > 1:
            m, b = np.polyfit(pair[sv], pair[ov], 1)
            xs = np.linspace(pair[sv].min(), pair[sv].max(), 50)
            ax.plot(xs, m * xs + b, color="red", linewidth=1)
        ax.set_xlabel(sv)
        ax.set_ylabel(ov)

plt.tight_layout()
plt.savefig("../output/01_sleep_vs_hrv_rhr_scatter.png", dpi=120)
plt.show()

## Findings

Ran on 270 days (2025-12-19 to 2026-09-14). Actual results from the correlation
table above:

- **No variable here strongly predicts either HRV or RHR** - every |r| is under
  0.19. This is a legitimate finding, not a failed analysis: sleep amount/
  consistency alone does not explain much of the night-to-night variation in
  this data.
- **sleep_efficiency_pct is the only variable significant against *both*
  outcomes** (HRV: r=-0.13, p=0.033; RHR: r=-0.12, p=0.044) - but the direction
  is counter-intuitive: *higher* sleep efficiency associates with *lower* HRV
  and *lower* RHR here. Needs a sanity check before trusting it (e.g. is a
  low-efficiency night usually a short, fragmented one during illness/travel,
  which would independently depress HRV and RHR for unrelated reasons?).
- **prior_5d_avg_sleep_hours vs HRV is the strongest single relationship**
  (r=-0.185, p=0.0025 - the lowest p-value in the whole table) but again
  counter to the naive hypothesis: more trailing sleep associates with *lower*
  next HRV, not higher. Notably, this same variable has essentially *zero*
  relationship with RHR (r=0.005, p=0.93) - so whatever this is, it's
  HRV-specific, not a general recovery effect. Possible read: this could be
  reverse causality - bigger training blocks or accumulated fatigue may drive
  both more sleep *and* lower HRV, rather than sleep amount lowering HRV
  directly. This one is worth its own follow-up notebook rather than a
  conclusion drawn here.
- **total_sleep_hours (last night specifically) and sleep_consistency_pct
  show no significant relationship with either outcome** in this sample - the
  two variables that seemed most obviously important going in turned out to
  be the ones with no signal.
- Simple takeaway for now: single-night sleep amount is *not* the driver of
  next-day HRV/RHR in this data. Whatever is moving HRV/RHR week to week,
  it is not captured by these four variables alone - a fair next step is
  incorporating exercise load (this notebook explicitly excluded it) rather
  than assuming sleep alone should carry the explanation.
